In [1]:
#!pip install paramiko
# Run this Notebook in Windsurf

In [2]:
import os
import posixpath
import time
from pathlib import Path

import pandas as pd
import paramiko
from dotenv import load_dotenv
from IPython import get_ipython
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager


In [3]:
load_dotenv(dotenv_path=Path('.env'), override=True)

sftp_host = os.getenv('SFTP_HOST')
sftp_login = os.getenv('SFTP_LOGIN')
sftp_password = os.getenv('SFTP_PASSWORD')
sftp_base_folder = os.getenv('SFTP_FOLDER')
tmdb_login = os.getenv('TMDB_LOGIN')
tmdb_password = os.getenv('TMDB_PASSWORD')

if not sftp_host or not sftp_login or sftp_password is None or not sftp_base_folder:
    raise ValueError('Missing SFTP configuration in .env: SFTP_HOST, SFTP_LOGIN, SFTP_PASSWORD, SFTP_FOLDER')

if not tmdb_login or not tmdb_password:
    raise ValueError('Missing TMDB configuration in .env: TMDB_LOGIN, TMDB_PASSWORD')


In [4]:
DATASETS = [
    {
        'name': 'movies',
        'remote_folder': 'wikidata-id-movie-fix',
        'local_folder': 'wikidata-id-movie-fix',
        'local_file_basename': 'wikidata-id-movie-fix',
        'entity_path': 'movie',
        'id_column': 'ID_MOVIE',
        'erase_column': 'ID_MOVIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngmovieidstart',
        'missing_entity_message': 'Colonne ID_MOVIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for movie {lngid}. Skipping.'
    },
    {
        'name': 'series',
        'remote_folder': 'wikidata-id-serie-fix',
        'local_folder': 'wikidata-id-serie-fix',
        'local_file_basename': 'wikidata-id-serie-fix',
        'entity_path': 'tv',
        'id_column': 'ID_SERIE',
        'erase_column': 'ID_SERIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngserieidstart',
        'missing_entity_message': 'Colonne ID_SERIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for serie {lngid}. Skipping.'
    },
    {
        'name': 'persons',
        'remote_folder': 'wikidata-id-person-fix',
        'local_folder': 'wikidata-id-person-fix',
        'local_file_basename': 'wikidata-id-person-fix',
        'entity_path': 'person',
        'id_column': 'ID_PERSON',
        'erase_column': 'ID_PERSON_ERASE_WIKIDATA_ID',
        'store_key': 'lngpersonidstart',
        'missing_entity_message': 'Colonne ID_PERSON manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for person {lngid}. Skipping.',
        'intgoingdown': True
    }
]


In [5]:
def download_new_csv_files(remote_folder, local_folder):
    remote_path = posixpath.join(sftp_base_folder, remote_folder)
    local_path = Path('./data') / local_folder
    local_path.mkdir(parents=True, exist_ok=True)

    downloaded_count = 0
    skipped_count = 0

    transport = paramiko.Transport((sftp_host, 22))
    transport.connect(username=sftp_login, password=sftp_password)
    sftp = paramiko.SFTPClient.from_transport(transport)

    try:
        for entry in sftp.listdir_attr(remote_path):
            if entry.filename in ('.', '..'):
                continue

            remote_file = posixpath.join(remote_path, entry.filename)
            local_file = local_path / entry.filename

            try:
                if (entry.st_mode & 0o170000) == 0o040000:
                    continue
            except Exception:
                pass

            if local_file.exists():
                skipped_count += 1
                continue

            sftp.get(remote_file, str(local_file))
            downloaded_count += 1
    finally:
        sftp.close()
        transport.close()

    print(f'SFTP folder: {remote_path}')
    print(f'Local folder: {local_path.resolve()}')
    print(f'Downloaded {downloaded_count} new file(s), skipped {skipped_count} existing file(s)')


def get_latest_csv(local_folder, local_file_basename):
    base_candidates = [Path('./data') / local_folder, Path('./data')]

    csv_candidates = []
    for base in base_candidates:
        if base.exists():
            csv_candidates.extend(base.glob(local_file_basename + '*.csv'))

    if not csv_candidates:
        raise FileNotFoundError(
            'No file found matching ' + local_file_basename + '*.csv in ./data or ./data/<local_folder>'
        )

    latest_csv = max(csv_candidates, key=lambda p: p.stat().st_mtime)
    print(f'Using latest CSV: {latest_csv}')
    return latest_csv


def set_store_value(store_key, value):
    ip = get_ipython()
    if ip is None:
        return

    ip.user_ns[store_key] = value
    ip.run_line_magic('store', store_key)


def init_driver():
    return webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))


def login_tmdb(driver):
    driver.get('https://www.themoviedb.org/login')
    time.sleep(10)

    username_field = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, 'username'))
    )
    username_field.clear()
    username_field.send_keys(tmdb_login)

    password_field = driver.find_element(By.NAME, 'password')
    password_field.clear()
    password_field.send_keys(tmdb_password)
    password_field.send_keys('\n')


def get_log_file_path():
    log_dir = Path('./data')
    log_dir.mkdir(parents=True, exist_ok=True)
    return log_dir / 'selenium-tmdb-wikidata_id.log'


def append_processed_log(content_type, record_id, wikidata_id):
    log_file_path = get_log_file_path()
    with log_file_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'{content_type};{record_id};{wikidata_id}\n')


def is_page_not_found(driver):
    elements = driver.find_elements(By.XPATH, "//h2[text()=\"Oops! We can't find the page you're looking for\"]")
    return len(elements) > 0


def set_wikidata_id(driver, entity_path, lngid, strwikidataid, timeout_message):
    driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')
    intpagefound = True
    try:
        if is_page_not_found(driver):
            print("Element 'Page 404' exists on the page.")
            intpagefound = False
        else:
            print("Element 'Page 404' does not exist on the page.")
    except NoSuchElementException:
        print("Element 'Page 404' does not exist on the page.")

    if intpagefound:
        xpath = "//button[span[contains(@class, 'glyphicons_v2') and contains(@class, 'plus') and contains(@class, 'svg')] and contains(., 'Create Translation')]"
        try:
            button = driver.find_element(By.XPATH, xpath)
            button.click()
            print("Button 'Create translation' clicked successfully.")
            time.sleep(2)
            driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')
        except Exception:
            print("Button 'Create translation' not found, translation already exists.")

        try:
            wikidata_id_field = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.NAME, 'wikidata_id'))
            )
        except TimeoutException:
            print(timeout_message.format(lngid=lngid))
            return False

        css_selector = 'span#wikidata_id_status.glyphicons_v2.locked.locked_status'
        elements = driver.find_elements(By.CSS_SELECTOR, css_selector)

        if len(elements) > 0:
            print("Element 'Locked' exists on the page.")
            return False

        wikidata_id_field.clear()
        wikidata_id_field.send_keys(strwikidataid)
        save_button = driver.find_element(By.XPATH, '//input[@value="Save"]')
        save_button.click()
        return True

    return False


def clear_wikidata_id(driver, entity_path, lngid):
    driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')

    if is_page_not_found(driver):
        print("Element 'Page 404' exists on the page. Skipping clear.")
        return False

    try:
        wikidata_id_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'wikidata_id'))
        )
    except TimeoutException:
        print(f'wikidata_id field not found for {entity_path} {lngid}. Skipping clear.')
        return False

    css_selector = 'span#wikidata_id_status.glyphicons_v2.locked.locked_status'
    elements = driver.find_elements(By.CSS_SELECTOR, css_selector)
    if len(elements) > 0:
        print("Element 'Locked' exists on the page. Skipping clear.")
        return False

    wikidata_id_field.clear()
    save_button = driver.find_element(By.XPATH, '//input[@value="Save"]')
    save_button.click()
    return True


def process_dataset(driver, dataset):
    latest_csv = get_latest_csv(dataset['local_folder'], dataset['local_file_basename'])
    data = pd.read_csv(str(latest_csv), sep=';', quotechar='"')
    print(data.shape)

    store_key = dataset['store_key']
    start_value = 0
    print(f'{store_key} = {start_value}')
    processed_count = 0

    if dataset.get('name') == 'persons' and not dataset.get('intgoingdown', True):
        rows = data.iloc[::-1].iterrows()
        comparator = lambda current_id, last_id: current_id < last_id
    else:
        rows = data.iterrows()
        comparator = lambda current_id, last_id: current_id > last_id

    for index, row in rows:
        print(f'Index: {index} ; processed: {processed_count}')

        if row[dataset['id_column']]:
            lngid = row[dataset['id_column']]
            if row['ID_WIKIDATA']:
                strwikidataid = row['ID_WIKIDATA']
                strtmdbidtoerase = row[dataset['erase_column']]
                if comparator(lngid, start_value):
                    if pd.isna(strtmdbidtoerase):
                        print('strtmdbidtoerase is NaN')
                    else:
                        print('strtmdbidtoerase is not NaN')
                        lngtmdbidtoerase = int(strtmdbidtoerase)
                        clear_wikidata_id(driver, dataset['entity_path'], lngtmdbidtoerase)

                    was_updated = set_wikidata_id(
                        driver,
                        dataset['entity_path'],
                        lngid,
                        strwikidataid,
                        dataset['timeout_message']
                    )
                    if was_updated:
                        append_processed_log(dataset['name'], lngid, strwikidataid)
                    processed_count += 1
                    start_value = lngid
                    set_store_value(store_key, start_value)
                    time.sleep(2)
            else:
                print(dataset['missing_wikidata_message'])
        else:
            print(dataset['missing_entity_message'])


In [6]:
for dataset in DATASETS:
    download_new_csv_files(dataset['remote_folder'], dataset['local_folder'])

driver = init_driver()
login_tmdb(driver)


SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-movie-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-movie-fix
Downloaded 7 new file(s), skipped 20 existing file(s)
SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-serie-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-serie-fix
Downloaded 7 new file(s), skipped 20 existing file(s)
SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-person-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-person-fix
Downloaded 7 new file(s), skipped 20 existing file(s)


In [7]:
process_dataset(driver, DATASETS[0])
process_dataset(driver, DATASETS[1])
process_dataset(driver, DATASETS[2])


Using latest CSV: data\wikidata-id-movie-fix\wikidata-id-movie-fix-20260625.csv
(78, 8)
lngmovieidstart = 0
Index: 0 ; processed: 0
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 1 ; processed: 1
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 2 ; processed: 2
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 3 ; processed: 3
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Sto

In [8]:
# Loop is finished so we display the home page
driver.get("https://www.themoviedb.org/")